# Resilient Plant · 05 · Sector Resilience Analysis

**Task 4 (reframed).** For each year 2014-2018, identify the bottom-quartile-margin metals firms (the 'shocked' cohort) and compare their Liquidity and Solvency profiles to the sector median. Use this empirical resilience profile to define the **Danger Zone** in percentile terms.

**Reframing rationale:** the brief's original 'supply & demand equilibrium' analysis required commodity price data not in the Kaggle dataset. This sector-resilience analysis answers the same underlying question — *what happens when shock hits?* — using 1,200+ real observations.

## Setup — auto-resolving paths

Run this cell first.

In [ ]:
# Auto-resolving path setup
# Folder layout expected:
#   Module 01/
#     ├── Track 03/
#     │   ├── dataset/                   (5 CSVs: 2014..2018_Financial_Data.csv)
#     │   └── resilient_plant/
#     │       ├── data/
#     │       ├── outputs/
#     │       ├── figures/
#     │       └── (notebooks live here, or in a notebooks/ subfolder)

from pathlib import Path

def find_project_root():
    p = Path.cwd().resolve()
    for parent in [p] + list(p.parents):
        if parent.name == "resilient_plant":
            return parent
        if (parent / "outputs").exists() and (parent / "data").exists() and (parent / "figures").exists():
            return parent
    return Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROJECT_ROOT = find_project_root()
DATASET_DIR  = PROJECT_ROOT.parent / "dataset"
DATA_DIR     = PROJECT_ROOT / "data"
OUTPUTS_DIR  = PROJECT_ROOT / "outputs"
FIGURES_DIR  = PROJECT_ROOT / "figures"

for d in [DATA_DIR, OUTPUTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset dir  : {DATASET_DIR}")
print(f"Data dir     : {DATA_DIR}")
print(f"Outputs dir  : {OUTPUTS_DIR}")
print(f"Figures dir  : {FIGURES_DIR}")

# Locate the 5 yearly CSVs — accept dataset/ or data/
YEARS = [2014, 2015, 2016, 2017, 2018]
csv_paths = {}
for y in YEARS:
    name = f"{y}_Financial_Data.csv"
    for folder in [DATASET_DIR, DATA_DIR]:
        if (folder / name).exists():
            csv_paths[y] = folder / name
            break
assert len(csv_paths) == 5, (
    f"Expected 5 CSVs (2014..2018_Financial_Data.csv) in {DATASET_DIR} or {DATA_DIR}; "
    f"found {sorted(csv_paths.keys())}"
)
print(f"Found 5 yearly CSVs : {sorted(csv_paths.keys())}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

clean = pd.read_csv(DATA_DIR / "metals_cleaned.csv")
print(f"Loaded {len(clean):,} metals company-years")


## Step 1 — Identify the shocked cohort each year (bottom-quartile op margin)

In [ ]:
shocked_rows = []
for y in sorted(clean["year"].unique()):
    sub = clean[clean["year"] == y].copy()
    threshold = sub["op_margin"].quantile(0.25)
    sub["cohort"] = np.where(sub["op_margin"] <= threshold, "Shocked (bottom 25%)", "Healthy (top 75%)")
    shocked_rows.append(sub)
labelled = pd.concat(shocked_rows, ignore_index=True)
print(f"Year-by-year shocked thresholds (op margin ≤ p25):")
print(labelled.groupby("year")["op_margin"].quantile(0.25).round(3))

# Tally
print(f"\nCohort sizes:")
print(labelled.groupby(["year", "cohort"]).size().unstack(fill_value=0))


## Step 2 — How do liquidity/solvency profiles differ?

In [ ]:
key_ratios = ["currentRatio", "quickRatio", "cashRatio",
              "debtRatio", "debtEquityRatio", "totalDebtToCapitalization",
              "returnOnAssets", "returnOnEquity"]

profile_table = (labelled.groupby("cohort")[key_ratios]
                 .median().round(3))
print("Median ratios — Shocked vs Healthy cohorts (pooled 2014-2018):")
print(profile_table.T.to_string())

profile_table.T.to_csv(OUTPUTS_DIR / "shocked_vs_healthy_profile.csv")


### Visualise the gap

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

groups = [
    ("Liquidity",     ["currentRatio", "quickRatio", "cashRatio"]),
    ("Solvency",      ["debtRatio", "debtEquityRatio", "totalDebtToCapitalization"]),
    ("Profitability", ["returnOnAssets", "returnOnEquity"]),
]

for ax, (group_name, cols) in zip(axes, groups):
    sub = profile_table[cols].T
    sub.plot(kind="bar", ax=ax, color=["#C00000", "#2E7D32"], width=0.7)
    ax.set_title(f"{group_name} · Shocked vs Healthy", fontweight="bold")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
    ax.legend(title="Cohort", fontsize=9)
    ax.axhline(0, color="grey", linewidth=0.7)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "shocked_vs_healthy.png", dpi=160, bbox_inches="tight")
plt.show()


## Step 3 — Define the Danger Zone using the empirical distribution

The stress test (Notebook 04) shows where NPV turns negative. Now we map those thresholds onto the 1,200+ observations: how many real metals firms have actually operated in those conditions?

In [ ]:
# Danger Zone thresholds from Notebook 04 (the conditions that turn NPV negative for the project)
# We'll use these test thresholds and check how often the sector hit them
danger_test_cases = [
    ("Op margin < 5%",                   labelled["op_margin"] < 0.05),
    ("Op margin < 4%",                   labelled["op_margin"] < 0.04),
    ("Op margin < 3%",                   labelled["op_margin"] < 0.03),
    ("Op margin < 0% (loss-making)",     labelled["op_margin"] < 0.00),
    ("Current ratio < 1.0 (illiquid)",   labelled["currentRatio"] < 1.0),
    ("Debt/Equity > 2.0 (leveraged)",    labelled["debtEquityRatio"] > 2.0),
]

results = []
total = len(labelled)
for desc, mask in danger_test_cases:
    n = mask.sum()
    pct = n / total * 100
    results.append({"condition": desc, "company_years": int(n), "pct_of_sector": round(pct, 1)})

danger = pd.DataFrame(results)
print("Empirical frequency of stress conditions (2014-2018, 1,200+ obs):")
print(danger.to_string(index=False))

danger.to_csv(OUTPUTS_DIR / "empirical_stress_frequency.csv", index=False)


## Step 4 — The 'Resilience Profile' — what distinguishes survivors?

Look at firms that were Shocked in Year N and ask: did they recover (margin in top 50% by Year N+1)? What ratios distinguished recoverers from non-recoverers?

In [ ]:
# For each company, find consecutive-year pairs where the firm was Shocked in Year N
recovery_records = []
firms = labelled["Unnamed: 0"].unique() if "Unnamed: 0" in labelled.columns else labelled.iloc[:, 0].unique()

# Use the first column as the company identifier (it's the ticker)
ticker_col = labelled.columns[0]
print(f"Using {ticker_col!r} as firm identifier")

for ticker, group in labelled.groupby(ticker_col):
    g = group.sort_values("year").reset_index(drop=True)
    for i in range(len(g) - 1):
        if g.loc[i, "cohort"] == "Shocked (bottom 25%)":
            n_year = g.loc[i, "year"]
            n1_row = g[g["year"] == n_year + 1]
            if len(n1_row) == 1:
                rec = n1_row.iloc[0]
                # "Recovered" = back to top 50% in next year
                year_n1 = rec["year"]
                median_n1 = labelled[labelled["year"] == year_n1]["op_margin"].median()
                recovered = rec["op_margin"] >= median_n1
                recovery_records.append({
                    "ticker": ticker,
                    "year_shocked": n_year,
                    "year_n1": year_n1,
                    "shocked_year_op_margin":   g.loc[i, "op_margin"],
                    "shocked_year_current_ratio": g.loc[i, "currentRatio"],
                    "shocked_year_debt_equity":   g.loc[i, "debtEquityRatio"],
                    "shocked_year_cash_ratio":    g.loc[i, "cashRatio"],
                    "next_year_op_margin":        rec["op_margin"],
                    "recovered":                  recovered,
                })

recovery = pd.DataFrame(recovery_records)
if len(recovery) > 0:
    print(f"\nFound {len(recovery)} shock→next-year transitions")
    print(f"Recovery rate: {recovery['recovered'].mean()*100:.1f}%")
    
    # Compare ratios at time of shock between recoverers and non-recoverers
    profile = recovery.groupby("recovered")[
        ["shocked_year_current_ratio", "shocked_year_debt_equity", "shocked_year_cash_ratio"]
    ].median().round(3)
    profile.index = ["Did NOT recover", "Recovered"]
    print(f"\nMedian ratios AT TIME OF SHOCK, by recovery outcome:")
    print(profile.T.to_string())
    
    profile.T.to_csv(OUTPUTS_DIR / "recovery_resilience_profile.csv")
else:
    print("No transition records found")


### Visualise the recovery profile

In [ ]:
if len(recovery) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    metrics = [
        ("shocked_year_current_ratio", "Current Ratio at time of shock"),
        ("shocked_year_debt_equity",   "Debt/Equity at time of shock"),
        ("shocked_year_cash_ratio",    "Cash Ratio at time of shock"),
    ]
    
    for ax, (col, title) in zip(axes, metrics):
        sub = recovery[[col, "recovered"]].copy()
        sub["status"] = sub["recovered"].map({True: "Recovered", False: "Did NOT recover"})
        sns.boxplot(data=sub, x="status", y=col, ax=ax,
                    palette={"Recovered": "#2E7D32", "Did NOT recover": "#C00000"})
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel(""); ax.set_ylabel("")
    
    plt.suptitle("Resilience Profile — what ratios at time of shock predicted recovery?",
                 fontweight="bold", y=1.02, fontsize=13)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "resilience_profile.png", dpi=160, bbox_inches="tight")
    plt.show()


## Step 5 — Where does the project sit in the empirical distribution?

The capital budgeting model uses op margin = 7.9% (sector median) as base case. Where does that sit in the empirical distribution? And where do the danger thresholds sit?

In [ ]:
op_margin_pooled = labelled["op_margin"].dropna()
project_thresholds = {
    "Project break-even (~12% op margin)":  0.120,
    "Sector top-quartile (~14%)":           0.140,
    "Sector median (7.9%)":                 0.079,
    "Sector bottom-quartile (~3%)":         0.030,
    "Loss-making threshold (0%)":           0.000,
}

print("Where does the project sit relative to the empirical sector distribution?")
print()
for name, val in project_thresholds.items():
    pct_below = (op_margin_pooled <= val).sum() / len(op_margin_pooled) * 100
    print(f"  {name:<40} {val:>6.1%}  ⇒  {pct_below:>5.1f}% of sector at-or-below this margin")
print()
print(f"Reading: the project's break-even op margin sits ABOVE the sector median —")
print(f"only top-quartile operators can justify this $75M CapEx at 10% WACC.")
print(f"At sector median, the project enters the Danger Zone immediately.")

# Save percentile-anchored thresholds
threshold_table = pd.DataFrame([
    {"threshold": name, "value": val,
     "sector_pct_at_or_below": round((op_margin_pooled <= val).sum() / len(op_margin_pooled) * 100, 1)}
    for name, val in project_thresholds.items()
])
threshold_table.to_csv(OUTPUTS_DIR / "project_vs_sector_distribution.csv", index=False)


✅ **Notebook 05 complete.** Move to `06_risk_audit_playbook.ipynb`.